# AI-Powered Technical Debt Quantification and Remediation
## API Notebook: Tool Exploration and Building Blocks

**Course:** DATA605 — Big Data Systems, Spring 2026  
**Team:** Akhil Kambhatla, Namratha Jeetendra, Hemanth Thulasiraman  
**Dataset:** The Technical Debt Dataset V2 (Lenarduzzi et al., 2019)

---

### What Is Technical Debt?

Technical debt is a concept from software engineering. When developers take shortcuts to ship code faster, they create problems that need to be fixed later. These shortcuts might be overly complex functions, duplicated code blocks, missing tests, or known bugs left unfixed. Just like financial debt, technical debt accumulates interest: the longer you wait to fix it, the more expensive it becomes.

SonarQube, one of the most widely used code analysis tools, classifies technical debt into three types:
- **Bugs:** Code that is objectively wrong and will likely cause failures
- **Code Smells:** Code that works but is hard to maintain, read, or extend
- **Vulnerabilities:** Code with security weaknesses

### Why Does This Matter?

According to the CISQ 2022 report, accumulated software technical debt in the US has grown to approximately $1.52 trillion (CISQ, 2022).

Recent work by Tornhill et al. (FSE 2025) introduced **ACE** (Augmented Code Engineering), a tool that uses LLMs to automatically refactor code and reduce technical debt. Their benchmarking study on over 100,000 real-world code issues found that the best-performing LLM generated functionally correct refactorings only 37% of the time when used without any validation. By adding a metric-driven validation pipeline that filters out bad suggestions, ACE achieved 98% precision on the remaining refactorings, though at the cost of only keeping 52% of the suggestions (the rest were discarded as unreliable).

However, ACE relies on proprietary tools (CodeScene, CodeHealth metric) and commercial LLMs. Our project investigates whether the same grounded approach works with fully open-source tools and a local code model that anyone can run for free.

### Our Research Question

**Can a fully open-source pipeline that combines ML-based fault prediction with local LLM code generation achieve effective technical debt remediation without commercial tools or API access?**

### What This Notebook Covers

This API notebook demonstrates each tool individually:

1. **Code metrics** with `radon` — measuring cyclomatic complexity
2. **Advanced metrics** with `lizard` — multi-language analysis
3. **Code smells** with `pylint` — detecting style and design violations
4. **AST analysis** with Python's `ast` module — understanding code structure
5. **Git mining** with `PyDriller` — extracting commit history
6. **Querying the Technical Debt Dataset** — SQL on real-world data
7. **Debt type classification** — training a classifier with scikit-learn
8. **Impact prediction** — training a regressor with XGBoost
9. **Code generation** — loading a pretrained code model
10. **Agent building blocks** — LangGraph tool-use patterns

The **Example notebook** combines all of these into an end-to-end pipeline and compares three approaches: SonarQube defaults, direct LLM prompting, and our grounded pipeline.

### Setup

All utility functions are defined in `ai_technical_debt_utils.py`. This notebook imports from that module and demonstrates each component individually.

In [2]:
# Standard imports.
import sys
import os
import warnings

# Add the project directory to the path so we can import our utils.
sys.path.insert(0, "/curr_dir")

# Suppress noisy warnings for cleaner notebook output.
warnings.filterwarnings("ignore")

# Project imports.
from ai_technical_debt_utils import *

print("All imports successful.")

All imports successful.


---
## Section 1: Code Complexity with `radon`

`radon` is a Python tool that computes code metrics, including:
- **Cyclomatic Complexity (CC):** the number of independent execution paths through a function. Higher CC means more branches, more test cases needed, and more cognitive load for developers. This is the same metric SonarQube computes for Java code in our Technical Debt Dataset.
- **Halstead Metrics:** measures of code "size" based on counting operators and operands.
- **Maintainability Index (MI):** a composite score (0-100) combining complexity, code length, and Halstead volume. Higher is better.

We use `radon` on Python code here to teach the concept. Later, in Section 6, we query the same metrics precomputed at scale on Java projects.

In [3]:
# We'll analyze two versions of the same function: one clean, one messy.
# This shows how complexity changes with coding style.

clean_code = '''
def calculate_grade(score):
    """Convert a numeric score to a letter grade."""
    if score >= 90:
        return "A"
    elif score >= 80:
        return "B"
    elif score >= 70:
        return "C"
    elif score >= 60:
        return "D"
    else:
        return "F"
'''

messy_code = '''
def process_student_data(students, include_inactive=False, sort_by=None,
                         min_score=0, max_score=100, normalize=False):
    """Process student records with multiple filtering and sorting options."""
    results = []
    for student in students:
        if not include_inactive and not student.get("active", True):
            continue
        score = student.get("score", 0)
        if score < min_score or score > max_score:
            continue
        if normalize and max_score != min_score:
            score = (score - min_score) / (max_score - min_score) * 100
        if score >= 90:
            grade = "A"
        elif score >= 80:
            grade = "B"
        elif score >= 70:
            grade = "C"
        elif score >= 60:
            grade = "D"
        else:
            grade = "F"
        result = {"name": student.get("name", "Unknown"), "grade": grade}
        if sort_by and sort_by in student:
            result["sort_key"] = student[sort_by]
        results.append(result)
    if sort_by:
        try:
            results.sort(key=lambda x: x.get("sort_key", ""))
        except TypeError:
            pass
    return results
'''

print("Code samples loaded.")
print(f"Clean function: {len(clean_code.strip().splitlines())} lines")
print(f"Messy function: {len(messy_code.strip().splitlines())} lines")

Code samples loaded.
Clean function: 12 lines
Messy function: 32 lines


In [8]:
# Compute cyclomatic complexity for both functions.
from radon.complexity import cc_visit, cc_rank

clean_results = cc_visit(clean_code)
messy_results = cc_visit(messy_code)

print("=== Clean Function ===")
for item in clean_results:
    print(f"  Function: {item.name}")
    print(f"  Cyclomatic Complexity: {item.complexity}")
    print(f"  Grade: {cc_rank(item.complexity)}")

print("\n=== Messy Function ===")
for item in messy_results:
    print(f"  Function: {item.name}")
    print(f"  Cyclomatic Complexity: {item.complexity}")
    print(f"  Grade: {cc_rank(item.complexity)}")

=== Clean Function ===
  Function: calculate_grade
  Cyclomatic Complexity: 5
  Grade: A

=== Messy Function ===
  Function: process_student_data
  Cyclomatic Complexity: 16
  Grade: C


Radon classifies complexity using letter grades:
- **A (1-5):** Low risk, simple block
- **B (6-10):** Low risk, well-structured and stable block
- **C (11-20):** Moderate, slightly complex block
- **D (21-30):** More than moderate, more complex block
- **E (31-40):** High, complex block, alarming
- **F (41+):** Very high, error-prone, unstable block

The clean function should score A (simple), while the messy function should score C (moderately complex) with its 16 branches and nested conditions.

In [7]:
# Compute Maintainability Index for both.
from radon.metrics import mi_visit

clean_mi = mi_visit(clean_code, multi=False)
messy_mi = mi_visit(messy_code, multi=False)

print("=== Maintainability Index (0-100, higher is better) ===")
print(f"  Clean function: {clean_mi:.1f}")
print(f"  Messy function: {messy_mi:.1f}")

if clean_mi > messy_mi:
    print(f"\n  The clean version is {clean_mi - messy_mi:.1f} points more maintainable.")

=== Maintainability Index (0-100, higher is better) ===
  Clean function: 65.3
  Messy function: 47.8

  The clean version is 17.6 points more maintainable.


---
## Section 2: Multi-Language Metrics with `lizard`

While `radon` only works on Python, `lizard` supports over 20 languages including Python, Java, C, C++, and JavaScript. This is relevant because the Technical Debt Dataset contains Java projects analyzed by SonarQube.

For each function, `lizard` computes:
- **NLOC:** Lines of code without comments
- **CCN:** Cyclomatic complexity number (same concept as radon's CC)
- **Token count:** Total number of code tokens (operators, identifiers, literals)
- **Parameter count:** Number of function parameters

We can also analyze Java code directly, which is useful since our dataset is entirely Java.

In [9]:
import lizard

# Analyze our Python functions using lizard.
# analyze_source_code requires a filename to identify the language.
clean_analysis = lizard.analyze_file.analyze_source_code(
    "clean.py", clean_code
)
messy_analysis = lizard.analyze_file.analyze_source_code(
    "messy.py", messy_code
)

print("=== Clean Function (Python) ===")
for func in clean_analysis.function_list:
    print(f"  Function: {func.name}")
    print(f"  NLOC: {func.nloc}")
    print(f"  CCN: {func.cyclomatic_complexity}")
    print(f"  Tokens: {func.token_count}")
    print(f"  Parameters: {func.parameter_count}")

print("\n=== Messy Function (Python) ===")
for func in messy_analysis.function_list:
    print(f"  Function: {func.name}")
    print(f"  NLOC: {func.nloc}")
    print(f"  CCN: {func.cyclomatic_complexity}")
    print(f"  Tokens: {func.token_count}")
    print(f"  Parameters: {func.parameter_count}")

=== Clean Function (Python) ===
  Function: calculate_grade
  NLOC: 11
  CCN: 5
  Tokens: 38
  Parameters: 1

=== Messy Function (Python) ===
  Function: process_student_data
  NLOC: 31
  CCN: 16
  Tokens: 198
  Parameters: 6


In [10]:
# lizard can also analyze Java code directly.
# This is relevant because our Technical Debt Dataset contains Java projects.
java_code = '''
public class Calculator {
    public double calculate(String operation, double a, double b) {
        if (operation.equals("add")) {
            return a + b;
        } else if (operation.equals("subtract")) {
            return a - b;
        } else if (operation.equals("multiply")) {
            return a * b;
        } else if (operation.equals("divide")) {
            if (b == 0) {
                throw new ArithmeticException("Division by zero");
            }
            return a / b;
        } else {
            throw new IllegalArgumentException("Unknown operation: " + operation);
        }
    }
}
'''

java_analysis = lizard.analyze_file.analyze_source_code(
    "Calculator.java", java_code
)

print("=== Java Function ===")
for func in java_analysis.function_list:
    print(f"  Function: {func.name}")
    print(f"  NLOC: {func.nloc}")
    print(f"  CCN: {func.cyclomatic_complexity}")
    print(f"  Tokens: {func.token_count}")
    print(f"  Parameters: {func.parameter_count}")

=== Java Function ===
  Function: Calculator::calculate
  NLOC: 16
  CCN: 6
  Tokens: 107
  Parameters: 3


Notice that `lizard` produces the same cyclomatic complexity as `radon` for the Python functions, but can also analyze Java code without any additional setup. This multi-language capability is what makes tools like `lizard` and SonarQube valuable for large organizations with mixed-language codebases.

In our Technical Debt Dataset, SonarQube computed these same metrics (complexity, NLOC, etc.) for all 154,000 commits across 31 Apache Java projects. The `SONAR_MEASURES` table contains these values precomputed at scale.

---
## Section 3: Code Smells with `pylint`

`radon` and `lizard` measure complexity (how many paths through the code). `pylint` is different: it checks whether the code follows good practices and coding standards. It detects things like:

- Unused variables and imports
- Missing docstrings
- Variable names that are too short or don't follow conventions
- Functions with too many arguments or local variables
- Duplicate code patterns

`pylint` scores code from 0 to 10, where 10 means fully compliant with coding standards. In the Technical Debt Dataset, SonarQube performs a similar role: it detects "code smells" (maintainability issues) and "bugs" (correctness issues) using its own rule set of over 500 rules for Java.

In [11]:
import tempfile
import os
from pylint.lint import Run
from pylint.reporters.text import TextReporter
import io

# Write the messy code to a temporary file for pylint to analyze.
with tempfile.NamedTemporaryFile(
    mode="w", suffix=".py", delete=False
) as f:
    f.write(messy_code)
    temp_path = f.name

# Run pylint and capture the output.
output = io.StringIO()
reporter = TextReporter(output)
try:
    Run(
        [temp_path, "--disable=C0114,C0115,C0116"],  # Disable module/class docstring warnings
        reporter=reporter,
        exit=False,
    )
except SystemExit:
    pass

# Print the results.
pylint_output = output.getvalue()
print(pylint_output)

# Clean up.
os.unlink(temp_path)

************* Module tmpk3j5xsvj
/tmp/tmpk3j5xsvj.py:2:0: R0913: Too many arguments (6/5) (too-many-arguments)
/tmp/tmpk3j5xsvj.py:2:0: R0917: Too many positional arguments (6/5) (too-many-positional-arguments)

-----------------------------------
Your code has been rated at 9.31/10




Notice something interesting: pylint gave the messy function 9.31/10 despite it having a cyclomatic complexity of 16 (grade C in radon). This is because **pylint and radon measure different things**. Pylint checks coding conventions and style. Radon measures structural complexity. Code can be perfectly formatted but deeply complex, or badly formatted but simple.

This is why our pipeline uses multiple metrics rather than relying on a single tool. In the Technical Debt Dataset, SonarQube combines both approaches: it has convention rules (similar to pylint) and complexity rules (similar to radon) in a single analysis.

The two issues pylint did find (`R0913: Too many arguments` and `R0917: Too many positional arguments`) are both in the **R (Refactor)** category, meaning pylint recommends restructuring the function. These correspond directly to what SonarQube would classify as a `CODE_SMELL`.

Each pylint message has a code like `C0301` or `R0913`. The first letter tells you the category:

- **C (Convention):** Style violations (naming, line length, formatting)
- **R (Refactor):** Code that should be restructured (too many arguments, too many branches)
- **W (Warning):** Potential problems that might cause bugs
- **E (Error):** Actual errors in the code
- **F (Fatal):** Errors that prevented pylint from analyzing the code

The **R** category (Refactor) is most relevant to technical debt: these are signals that code needs structural improvement. Messages like `R0913: Too many arguments` or `R0912: Too many branches` directly correspond to the kinds of issues SonarQube flags as code smells in our dataset.

---
## Section 4: Code Structure with Python's `ast` Module

The `ast` (Abstract Syntax Tree) module is built into Python. It parses source code into a tree structure that represents the code's logical organization: which functions exist, how deeply nested they are, what operations they perform.

Unlike radon, lizard, and pylint (which are third-party tools), `ast` gives you raw access to the code's structure. This is useful for building custom metrics that standard tools don't provide, such as:

- Counting the number of functions and classes in a file
- Measuring maximum nesting depth (how many levels of indentation)
- Detecting function calls to specific libraries
- Finding hardcoded values that should be constants

In [12]:
import ast

# Parse the messy code into an AST.
tree = ast.parse(messy_code)

# Walk the tree and count different node types.
node_counts = {}
for node in ast.walk(tree):
    name = type(node).__name__
    node_counts[name] = node_counts.get(name, 0) + 1

# Show the most common node types.
print("=== AST Node Counts (top 10) ===")
sorted_counts = sorted(node_counts.items(), key=lambda x: -x[1])
for name, count in sorted_counts[:10]:
    print(f"  {name}: {count}")

=== AST Node Counts (top 10) ===
  Name: 44
  Load: 42
  Constant: 27
  Store: 11
  Assign: 10
  If: 9
  Compare: 8
  arg: 7
  Call: 6
  Attribute: 6


In [13]:
def analyze_code_structure(source_code):
    """
    Extract structural metrics from Python source code using the ast module.
    
    Returns a dictionary with:
    - num_functions: number of function definitions
    - num_classes: number of class definitions
    - max_depth: maximum nesting depth of control structures
    - num_branches: total number of if/elif/else branches
    - num_loops: total number of for/while loops
    - num_try_except: total number of try/except blocks
    """
    tree = ast.parse(source_code)
    
    metrics = {
        "num_functions": 0,
        "num_classes": 0,
        "num_branches": 0,
        "num_loops": 0,
        "num_try_except": 0,
    }
    
    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef):
            metrics["num_functions"] += 1
        elif isinstance(node, ast.ClassDef):
            metrics["num_classes"] += 1
        elif isinstance(node, (ast.If,)):
            metrics["num_branches"] += 1
        elif isinstance(node, (ast.For, ast.While)):
            metrics["num_loops"] += 1
        elif isinstance(node, ast.Try):
            metrics["num_try_except"] += 1
    
    return metrics

print("=== Clean Function ===")
clean_metrics = analyze_code_structure(clean_code)
for key, value in clean_metrics.items():
    print(f"  {key}: {value}")

print("\n=== Messy Function ===")
messy_metrics = analyze_code_structure(messy_code)
for key, value in messy_metrics.items():
    print(f"  {key}: {value}")

=== Clean Function ===
  num_functions: 1
  num_classes: 0
  num_branches: 4
  num_loops: 0
  num_try_except: 0

=== Messy Function ===
  num_functions: 1
  num_classes: 0
  num_branches: 9
  num_loops: 1
  num_try_except: 1


The AST analysis confirms what radon and lizard told us: the messy function has more branches, more loops, and a try/except block that the clean function lacks. 

Custom AST analysis like this is valuable when you need metrics that standard tools don't provide. For example, you could write an AST visitor that counts how many external library calls a function makes (a measure of coupling), or that detects deeply nested loops (a performance concern).

In our pipeline, we use precomputed metrics from SonarQube rather than building custom AST analyzers. But understanding how AST analysis works helps explain what tools like SonarQube are doing under the hood.

---
## Section 5: Git Mining with `PyDriller`

`PyDriller` is the tool the Technical Debt Dataset authors used to extract commit history from 33 Apache Java projects (Lenarduzzi et al., 2019). It walks through a Git repository's history and extracts structured data about each commit: who made it, when, what files changed, how many lines were added or removed, and the actual diffs.

This is how the `GIT_COMMITS` and `GIT_COMMITS_CHANGES` tables in our dataset were populated. Here we demonstrate PyDriller on a live repository to show how it works.

In [16]:
from pydriller import Repository

# Mine the last 5 commits from this project's own repository.
# The repo is mounted at /git_root inside Docker.
repo_path = "/git_root"

commits = []
for commit in Repository(repo_path, order="reverse").traverse_commits():
    commits.append({
        "hash": commit.hash[:8],
        "author": commit.author.name,
        "date": commit.author_date.strftime("%Y-%m-%d %H:%M"),
        "message": commit.msg.split("\n")[0][:80],
        "files_changed": commit.files,
        "insertions": commit.insertions,
        "deletions": commit.deletions,
    })
    if len(commits) >= 5:
        break

import pandas as pd
commits_df = pd.DataFrame(commits)
print(f"Showing the 5 most recent commits from {repo_path}:\n")
commits_df

INFO:pydriller.repository:Analyzing git repository in /git_root
INFO:pydriller.repository:Commit #8867ba6fcea775947dd23d024c7c5368a8a742ad in 2026-04-17 16:27:43-04:00 from Akhil
INFO:pydriller.repository:Commit #fb2c04ca0b6c2a330cca597f0d12852dc2523e7c in 2026-04-16 21:55:31-04:00 from Akhil
INFO:pydriller.repository:Commit #435acb350752ff93af690bef05804f93675c43ae in 2026-03-26 15:51:30-04:00 from Akhil
INFO:pydriller.repository:Commit #6c5ff2418bf6a0ab79ef54ddf4e219b3b75e3358 in 2026-03-05 15:57:42-05:00 from Damian Calabresi
INFO:pydriller.repository:Commit #06338c563d8067a59fc14eb7ca3bd44cfb618e7a in 2026-03-05 15:57:19-05:00 from Damian Calabresi


Showing the 5 most recent commits from /git_root:



,hash,author,date,message,files_changed,insertions,deletions
0,8867ba6f,Akhil,2026-04-17 16:27,[Module 1] Add data loading and feature engine...,2,583,44
1,fb2c04ca,Akhil,2026-04-16 21:55,"[Setup] Rename template files, configure Docke...",9,156,37
2,435acb35,Akhil,2026-03-26 15:51,Added Project Template for AI Powered Technica...,23,1725,0
3,6c5ff241,Damian Calabresi,2026-03-05 15:57,Tutor task322 ax multi objective optimization ...,1,8,50
4,06338c56,Damian Calabresi,2026-03-05 15:57,Blog post: Introduction To Bayesian Optimizati...,1,7,1


In [17]:
# Look at the details of one commit: which files changed and how.
for commit in Repository(repo_path, order="reverse").traverse_commits():
    if commit.files > 0:
        print(f"Commit: {commit.hash[:8]}")
        print(f"Author: {commit.author.name}")
        print(f"Message: {commit.msg.split(chr(10))[0]}")
        print(f"\nFiles modified:")
        for mod_file in commit.modified_files:
            print(f"  {mod_file.filename}")
            print(f"    Lines added: {mod_file.added_lines}")
            print(f"    Lines removed: {mod_file.deleted_lines}")
            print(f"    Change type: {mod_file.change_type.name}")
            if mod_file.complexity is not None:
                print(f"    Complexity: {mod_file.complexity}")
        break  # Only show one commit.

INFO:pydriller.repository:Analyzing git repository in /git_root
INFO:pydriller.repository:Commit #8867ba6fcea775947dd23d024c7c5368a8a742ad in 2026-04-17 16:27:43-04:00 from Akhil


Commit: 8867ba6f
Author: Akhil
Message: [Module 1] Add data loading and feature engineering functions in utils

Files modified:
  .gitignore
    Lines added: 1
    Lines removed: 0
    Change type: ADD
  ai_technical_debt_utils.py
    Lines added: 582
    Lines removed: 44
    Change type: MODIFY
    Complexity: 41


PyDriller gives us structured access to the same data that populates the `GIT_COMMITS` and `GIT_COMMITS_CHANGES` tables in the Technical Debt Dataset. The key fields are:

- **Files changed, insertions, deletions:** Measures the size of each commit
- **Change type:** Whether a file was Added, Modified, Deleted, or Renamed
- **Complexity:** PyDriller can compute cyclomatic complexity per file using `lizard` internally

In the Technical Debt Dataset, this information was collected for all 153,994 commits across 31 projects. In the next section, we query that precomputed data directly.

---
## Section 6: Querying the Technical Debt Dataset

In Sections 1-5, we demonstrated individual tools on small code samples. Now we connect to the actual dataset used in our research: the Technical Debt Dataset V2 (Lenarduzzi et al., 2019).

This dataset contains the output of running SonarQube, Ptidej, Refactoring Miner, and the SZZ algorithm across 31 Apache Java projects. The data is stored as a SQLite database with 10 tables covering:

- **Code metrics** (complexity, coverage, duplication) per commit
- **Detected issues** (bugs, code smells, vulnerabilities) with severity and remediation effort
- **Refactoring operations** applied by developers
- **Fault-inducing commits** identified by the SZZ algorithm
- **Jira issue reports** from the project tracker

All data loading functions are defined in `ai_technical_debt_utils.py`.

In [18]:
# Connect to the database and get an overview.
conn = connect_to_database("/curr_dir/data/td_V2.db")
summary = get_dataset_summary(conn)
summary

INFO:ai_technical_debt_utils:Connecting to database: /curr_dir/data/td_V2.db
INFO:ai_technical_debt_utils:Dataset summary:
                TABLE_NAME  ROW_COUNT
               GIT_COMMITS     153994
       GIT_COMMITS_CHANGES    1142878
            SONAR_MEASURES      66711
              SONAR_ISSUES    1024614
            SONAR_ANALYSIS      67550
               SONAR_RULES       1819
         REFACTORING_MINER     362253
               JIRA_ISSUES      61402
SZZ_FAULT_INDUCING_COMMITS      52428
                  PROJECTS         31


,TABLE_NAME,ROW_COUNT
0,GIT_COMMITS,153994
1,GIT_COMMITS_CHANGES,1142878
2,SONAR_MEASURES,66711
3,SONAR_ISSUES,1024614
4,SONAR_ANALYSIS,67550
5,SONAR_RULES,1819
6,REFACTORING_MINER,362253
7,JIRA_ISSUES,61402
8,SZZ_FAULT_INDUCING_COMMITS,52428
9,PROJECTS,31


In [19]:
# List all projects in the dataset.
projects = get_projects(conn)
print(f"Total projects: {len(projects)}\n")
projects[["PROJECT_ID", "PROJECT_KEY"]]

INFO:ai_technical_debt_utils:Loaded 31 projects


Total projects: 31



,PROJECT_ID,PROJECT_KEY
0,org.apache:batik,batik
1,org.apache:bcel,commons-bcel
2,org.apache:beanutils,commons-beanutils
3,org.apache:cocoon,cocoon
4,org.apache:codec,commons-codec
5,org.apache:collections,commons-collections
6,org.apache:commons-cli,commons-cli
7,org.apache:commons-exec,commons-exec
8,org.apache:commons-fileupload,commons-fileupload
9,org.apache:commons-io,commons-io


### Exploring Code Metrics

The `SONAR_MEASURES` table contains 30+ code metrics per commit snapshot. Let's look at one project to understand what the data looks like.

In [20]:
# Load metrics for a medium-sized project.
measures = get_sonar_measures(conn, "org.apache:commons-io")
print(f"Shape: {measures.shape}")
print(f"\nNumeric columns available:")

# Show a subset of the most important metrics.
key_metrics = [
    "COMPLEXITY", "COGNITIVE_COMPLEXITY", "COVERAGE",
    "DUPLICATED_LINES_DENSITY", "NCLOC", "FUNCTIONS",
    "CLASSES",
]
available = [c for c in key_metrics if c in measures.columns]
measures[available].describe().round(2)

INFO:ai_technical_debt_utils:Loaded 1910 SONAR_MEASURES rows for project org.apache:commons-io


Shape: (1910, 240)

Numeric columns available:


,COMPLEXITY,COGNITIVE_COMPLEXITY,COVERAGE,NCLOC,FUNCTIONS,CLASSES
count,1908.00,1908.00,1908.0,1908.00,1908.00,1908.00
mean,2654.92,1725.21,0.0,17065.10,1643.60,164.10
std,1232.13,837.56,0.0,8010.86,754.55,71.39
min,55.00,31.00,0.0,312.00,34.00,4.00
25%,1470.00,926.00,0.0,9489.00,907.00,89.00
50%,2781.00,1808.50,0.0,18162.00,1777.00,190.00
75%,3797.00,2519.00,0.0,24194.25,2358.00,228.00
max,4491.00,2858.00,0.0,32945.00,2702.00,272.00


### Exploring Technical Debt Issues

The `SONAR_ISSUES` table contains over 1 million individual issues detected by SonarQube and Ptidej. Each issue has a type, severity, and estimated remediation effort.

In [21]:
# Load issues for the same project.
issues = get_sonar_issues(conn, "org.apache:commons-io")
print(f"Total issues for commons-io: {len(issues)}\n")

# Distribution by type.
print("Issues by TYPE:")
print(issues["TYPE"].value_counts().to_string())

print("\n\nIssues by SEVERITY:")
print(issues["SEVERITY"].value_counts().to_string())

INFO:ai_technical_debt_utils:Loaded 4000 SONAR_ISSUES rows for project org.apache:commons-io


Total issues for commons-io: 4000

Issues by TYPE:
TYPE
CODE_SMELL       3888
BUG                96
VULNERABILITY      16


Issues by SEVERITY:
SEVERITY
MAJOR       2001
MINOR       1198
CRITICAL     589
INFO         200
BLOCKER       12


### Fault-Inducing Commits and the SZZ Algorithm

The `SZZ_FAULT_INDUCING_COMMITS` table links Jira bug reports to the specific commits that introduced them. This was computed using the SZZ algorithm, which traces bug-fixing commits back through Git history to find the original fault-inducing commit.

This data is the ground truth for our impact prediction model: we want to predict which commits will turn out to be fault-inducing based on their code metrics at the time of the commit.

In [22]:
# Load fault-inducing commit data.
faults = get_fault_inducing_commits(conn, "org.apache:commons-io")
print(f"Fault-inducing commit records: {len(faults)}")
print(f"Unique fault-inducing commits: {faults['FAULT_INDUCING_COMMIT_HASH'].nunique()}")
print(f"Unique fault-fixing commits: {faults['FAULT_FIXING_COMMIT_HASH'].nunique()}")

INFO:ai_technical_debt_utils:Loaded 1452 fault-inducing commit records for project org.apache:commons-io


Fault-inducing commit records: 1452
Unique fault-inducing commits: 597
Unique fault-fixing commits: 126


### Building the Feature Matrix

Our `build_full_feature_matrix()` function joins the metrics, issues, and fault labels into a single dataframe ready for machine learning. Each row is one commit with all its metrics as features and a binary label indicating whether it introduced a fault.

In [24]:
# Build the full feature matrix for commons-io.
feature_matrix = build_full_feature_matrix(conn, "org.apache:commons-io")
print(f"Feature matrix shape: {feature_matrix.shape}")
print(f"Fault-inducing commits: {feature_matrix['IS_FAULT_INDUCING'].sum()} "
      f"/ {len(feature_matrix)} "
      f"({100 * feature_matrix['IS_FAULT_INDUCING'].mean():.1f}%)")

# Show the class balance.
print(f"\nClass distribution:")
print(feature_matrix["IS_FAULT_INDUCING"].value_counts().to_string())

INFO:ai_technical_debt_utils:Project org.apache:commons-io: 1910 commits with metrics
INFO:ai_technical_debt_utils:Loaded 1452 fault-inducing commit records for project org.apache:commons-io
INFO:ai_technical_debt_utils:Project org.apache:commons-io: 597 unique fault-inducing commits
INFO:ai_technical_debt_utils:Project org.apache:commons-io: 484 / 1910 commits are fault-inducing (25.3%)
INFO:ai_technical_debt_utils:Project org.apache:commons-io: issue counts for 402 commits
INFO:ai_technical_debt_utils:Project org.apache:commons-io: full feature matrix has 1910 rows and 252 columns


Feature matrix shape: (1910, 252)
Fault-inducing commits: 484 / 1910 (25.3%)

Class distribution:
IS_FAULT_INDUCING
0    1426
1     484


**Note:** All the queries above are for a single project (`commons-io`) as a demonstration. The full dataset contains 31 Apache projects with a combined 153,994 commits, 1,024,614 issues, and 52,428 fault-inducing commit records. In the Example notebook, we build feature matrices across multiple projects for training and evaluation.